# num2words

num2words 0.5.14 turns numbers into words in about 60 languages: cardinals, ordinals, years and currency amounts, as a
library and as the `num2words` command. These cells check known answers in several languages and modes, the two ways it
refuses bad input, the command-line tool, and the use that made us package it: transformers' SmolVLM processor writes the
number of video frames in words with it.

In [ ]:
# Example 1: cardinals in several languages and the ordinal and year modes, against known answers.
from num2words import num2words

print(num2words(42), "|", num2words(42, lang="fr"), "|", num2words(42, lang="de"), "|", num2words(42, lang="es"))
assert num2words(42) == "forty-two"
assert num2words(42, lang="fr") == "quarante-deux"
assert num2words(42, lang="de") == "zweiundvierzig"
assert num2words(42, lang="es") == "cuarenta y dos"
assert num2words(7, to="ordinal") == "seventh"
assert num2words(7, to="ordinal_num") == "7th"
assert num2words(1999, to="year") == "nineteen ninety-nine"

In [ ]:
# Example 2: currency amounts and floating point numbers, exact strings.
from num2words import num2words

usd = num2words(1234.56, to="currency", currency="USD")
eur = num2words(3.5, to="currency", currency="EUR", lang="fr")
flt = num2words(3.14)
print(usd, "|", eur, "|", flt)
assert usd == "one thousand, two hundred and thirty-four dollars, fifty-six cents"
assert eur == "trois euros et cinquante centimes"
assert flt == "three point one four"

In [ ]:
# Example 3: negative numbers, very large numbers and zero.
from num2words import num2words

print(num2words(-3), "|", num2words(10**12), "|", num2words(0), "|", num2words(1000000, lang="de"))
assert num2words(-3) == "minus three"
assert num2words(10**12) == "one trillion"
assert num2words(0) == "zero"
assert num2words(1000000, lang="de") == "eine Million"

In [ ]:
# Example 4: bad input is refused, not guessed at: an unknown language and an unknown mode.
from num2words import num2words

for kwargs in ({"lang": "xx_NOPE"}, {"to": "nonsense"}):
    try:
        num2words(5, **kwargs)
    except NotImplementedError as e:
        print("refused:", kwargs, type(e).__name__)
    else:
        raise AssertionError(f"num2words accepted {kwargs}")

In [ ]:
# Example 5: the command-line tool, and the SmolVLM processor path that made us package this.
import shutil, subprocess

out = subprocess.run([shutil.which("num2words"), "42", "--lang", "de"], capture_output=True, text=True, check=True).stdout.strip()
print("cli:", out)
assert out == "zweiundvierzig"

from transformers.utils import is_num2words_available
from transformers.models.smolvlm.processing_smolvlm import DEFAULT_VIDEO_INTRO, num2words as smolvlm_num2words

assert is_num2words_available() is True
assert smolvlm_num2words is not None, "transformers' SmolVLM processor did not find num2words"
intro = DEFAULT_VIDEO_INTRO.format(frame_count=smolvlm_num2words(8), video_duration="0:00:03")
print(intro)
assert "eight frames" in intro